<div id="singlestore-header" style="display: flex; background-color: rgba(124, 195, 235, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/book-open-cover.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Develop Safely with Database Branching</h1>
    </div>
</div>

## What is Database Branching:

Database branching in SingleStore enables the creation of an isolated copy of an existing database—sharing its history but functioning independently—so you can insert, update, or delete data without impacting the parent database.

### Benefits:
- It allows rapid creation of isolated environments for prototyping or testing schema changes without disrupting production, and supports point-in-time recovery by branching to a specific timestamp, both improving speed and reducing risk.

- Traditional SQL systems required full volume-level clones or long database restores to enable branching or sandboxing. SingleStore integrates branching as a standard feature, enabling rapid, low-cost sandboxing and developer productivity without heavy SANs or long wait times.

### Use Cases:
##### 1. Development and Testing-
- Instantly spin up isolated branches with the latest replica of the production database or at a specific point in time.
##### 2. Performance Tuning-
- You can evaluate multiple versions of your application on a branch, isolating and resolving any issues while the production database remains active.
##### 3. Database Maintenance-
- create a new database branch from the original database to rename an existing database.



### Architecture Diagram
TBD
TBD
TBD
TBD

#### Prerequisite: generate the Digital Marketing dataset
The Digital Marketing dataset can be generated using the website here: [digital-marketing.labs.singlestore.com](https://digital-marketing.labs.singlestore.com/)

1. Make sure your Standard or Premium workspace is running.
2. Open the website and fill in the connection form: your workspace host with `https://` in front (find it under **Connect** in the portal), username `admin`, your workspace password, and database name `martech`. Click **Connect**.
3. Click **Setup Database**. It creates the `martech` database and starts streaming shopper locations and purchases into it. Keep the **Simulator** switch on.
4. Under **Locations**, tick the cities you want. Each city adds ad campaigns, so more cities give the notebook more ads to search.
5. Wait until **Ad Campaigns** shows a number above 0, then come back here.
If the connection fails, check that your workspace's firewall allows your IP address.

The notebook only reads from `martech`. If you skip this step, it loads the same data from the dataset's public S3 bucket instead.

# OR

### Create Database and Tables
Connect to your SingleStore workspace and run the following SQL commands to create a database named tpch and eight associated tables.

In [8]:
%%sql
DROP DATABASE IF EXISTS tpch;
CREATE DATABASE tpch;
USE tpch;

CREATE TABLE `customer` (
  `c_custkey` int(11) NOT NULL,
  `c_name` varchar(25) COLLATE utf8mb4_bin NOT NULL,
  `c_address` varchar(40) COLLATE utf8mb4_bin NOT NULL,
  `c_nationkey` int(11) NOT NULL,
  `c_phone` char(15) COLLATE utf8mb4_bin NOT NULL,
  `c_acctbal` decimal(15,2) NOT NULL,
  `c_mktsegment` char(10) COLLATE utf8mb4_bin NOT NULL,
  `c_comment` varchar(117) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`c_custkey`) UNENFORCED RELY,
  SHARD KEY (`c_custkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `lineitem` (
  `l_orderkey` bigint(11) NOT NULL,
  `l_partkey` int(11) NOT NULL,
  `l_suppkey` int(11) NOT NULL,
  `l_linenumber` int(11) NOT NULL,
  `l_quantity` decimal(15,2) NOT NULL,
  `l_extendedprice` decimal(15,2) NOT NULL,
  `l_discount` decimal(15,2) NOT NULL,
  `l_tax` decimal(15,2) NOT NULL,
  `l_returnflag` char(1) COLLATE utf8mb4_bin NOT NULL,
  `l_linestatus` char(1) COLLATE utf8mb4_bin NOT NULL,
  `l_shipdate` date NOT NULL,
  `l_commitdate` date NOT NULL,
  `l_receiptdate` date NOT NULL,
  `l_shipinstruct` char(25) COLLATE utf8mb4_bin NOT NULL,
  `l_shipmode` char(10) COLLATE utf8mb4_bin NOT NULL,
  `l_comment` varchar(44) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`l_orderkey`, `l_linenumber`) UNENFORCED RELY,
  SHARD KEY (`l_orderkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `nation` (
  `n_nationkey` int(11) NOT NULL,
  `n_name` char(25) COLLATE utf8mb4_bin NOT NULL,
  `n_regionkey` int(11) NOT NULL,
  `n_comment` varchar(152) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`n_nationkey`) UNENFORCED RELY,
  SHARD KEY (`n_nationkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `orders` (
  `o_orderkey` bigint(11) NOT NULL,
  `o_custkey` int(11) NOT NULL,
  `o_orderstatus` char(1) COLLATE utf8mb4_bin NOT NULL,
  `o_totalprice` decimal(15,2) NOT NULL,
  `o_orderdate` date NOT NULL,
  `o_orderpriority` char(15) COLLATE utf8mb4_bin NOT NULL,
  `o_clerk` char(15) COLLATE utf8mb4_bin NOT NULL,
  `o_shippriority` int(11) NOT NULL,
  `o_comment` varchar(79) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`o_orderkey`) UNENFORCED RELY,
  SHARD KEY (`o_orderkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `part` (
  `p_partkey` int(11) NOT NULL,
  `p_name` varchar(55) COLLATE utf8mb4_bin NOT NULL,
  `p_mfgr` char(25) COLLATE utf8mb4_bin NOT NULL,
  `p_brand` char(10) COLLATE utf8mb4_bin NOT NULL,
  `p_type` varchar(25) COLLATE utf8mb4_bin NOT NULL,
  `p_size` int(11) NOT NULL,
  `p_container` char(10) COLLATE utf8mb4_bin NOT NULL,
  `p_retailprice` decimal(15,2) NOT NULL,
  `p_comment` varchar(23) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`p_partkey`) UNENFORCED RELY,
  SHARD KEY (`p_partkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `partsupp` (
  `ps_partkey` int(11) NOT NULL,
  `ps_suppkey` int(11) NOT NULL,
  `ps_availqty` int(11) NOT NULL,
  `ps_supplycost` decimal(15,2) NOT NULL,
  `ps_comment` varchar(199) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`ps_partkey`,`ps_suppkey`) UNENFORCED RELY,
  SHARD KEY(`ps_partkey`),
  KEY (`ps_partkey`,`ps_suppkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `region` (
  `r_regionkey` int(11) NOT NULL,
  `r_name` char(25) COLLATE utf8mb4_bin NOT NULL,
  `r_comment` varchar(152) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`r_regionkey`) UNENFORCED RELY,
  SHARD KEY (`r_regionkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE `supplier` (
  `s_suppkey` int(11) NOT NULL,
  `s_name` char(25) COLLATE utf8mb4_bin NOT NULL,
  `s_address` varchar(40) COLLATE utf8mb4_bin NOT NULL,
  `s_nationkey` int(11) NOT NULL,
  `s_phone` char(15) COLLATE utf8mb4_bin NOT NULL,
  `s_acctbal` decimal(15,2) NOT NULL,
  `s_comment` varchar(101) COLLATE utf8mb4_bin NOT NULL,
  UNIQUE KEY pk (`s_suppkey`) UNENFORCED RELY,
  SHARD KEY (`s_suppkey`) USING CLUSTERED COLUMNSTORE
);

8 rows affected.

1 rows affected.

++
||
++
++

### Load Data with Pipelines
This part of the tutorial shows how to ingest TPC-H data from a public AWS S3 bucket into the SingleStore database using pipelines.

In [ ]:
%%sql
USE tpch;

CREATE OR REPLACE PIPELINE tpch_100_lineitem
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/lineitem/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE lineitem
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_customer
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/customer/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE customer
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_nation
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/nation/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE nation
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_orders
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/orders/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE orders
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_part
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/part/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE part
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_partsupp
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/partsupp/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE partsupp
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_region
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/region/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE region
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

CREATE OR REPLACE PIPELINE tpch_100_supplier
    AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/supplier/'
    config '{"region":"us-east-1"}'
    SKIP DUPLICATE KEY ERRORS
    INTO TABLE supplier
    FIELDS TERMINATED BY '|'
    LINES TERMINATED BY '|\n';

Run the following SQL commands to start the pipelines:

In [ ]:
%%sql
USE tpch;
START ALL PIPELINES;

## Check pipeline status
-- SELECT * FROM information_schema.pipelines_files WHERE pipeline_name = "tpch_100_lineitem";

#### Check table count

In [17]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the workspace/database dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT table_name FROM information_schema.tables "
                    "WHERE table_schema = DATABASE() AND table_type = 'BASE TABLE' ORDER BY table_name")
        tables = [r[0] for r in cur.fetchall()]

        pt = PrettyTable(['table_name', 'singlestore'])
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM `{t}`")
            pt.add_row([t, cur.fetchone()[0]])

print(pt)

+------------+-------------+
| table_name | singlestore |
+------------+-------------+
|  customer  |   15000000  |
|  lineitem  |  538837902  |
|   nation   |      25     |
|   orders   |  150000000  |
|    part    |   20000000  |
|  partsupp  |   80000000  |
|   region   |      5      |
|  supplier  |   1000000   |
+------------+-------------+


### Creating a Branch
 1. Using SingleStore Helios portal
 2. Using SQL

In [19]:
%%sql
## Creating Database Branch using SQL:

ATTACH DATABASE tpch AS tpch_branch;

1 rows affected.

++
||
++
++

Branching can use PITR and this requires the database to be deployed using an Enterprise project. 

Think of a scenario when user accidently truncate Orders table. You can recover the data by simply creating a branch of your database at a point in time before the bad query was executed.

In [20]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the workspace dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("USE tpch_branch")
        cur.execute("SELECT DATABASE()");           db = cur.fetchone()[0]
        cur.execute("SELECT NOW()");                now = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM orders"); before = cur.fetchone()[0]
        cur.execute("DELETE FROM orders LIMIT 1000");          truncated = cur.rowcount
        conn.commit()
        cur.execute("SELECT COUNT(*) FROM orders"); after = cur.fetchone()[0]
        cur.execute("SELECT NOW()");                end = cur.fetchone()[0]

pt = PrettyTable(['Database', 'Time', 'Before delete', 'Rows deleted', 'After delete', 'End Time'])
pt.add_row([db, now, f"{before:,}", f"{truncated:,}", f"{after:,}", end])
print(pt)

+-------------+---------------------+---------------+--------------+--------------+---------------------+
|   Database  |         Time        | Before delete | Rows deleted | After delete |       End Time      |
+-------------+---------------------+---------------+--------------+--------------+---------------------+
| tpch_branch | 2026-09-29 22:09:08 |  150,000,000  |    1,000     | 149,999,000  | 2026-09-29 22:09:11 |
+-------------+---------------------+---------------+--------------+--------------+---------------------+


In [24]:
%%sql
%%sql
USE tpch_branch;

SELECT count(*) FROM orders;

1 rows affected.

count(*)
149999000


In [22]:
%%sql

ATTACH DATABASE tpch_branch AS tpch_branch_new AT TIME '2026-09-29 22_08_58';

1 rows affected.

++
||
++
++

In [25]:
%%sql
USE tpch_branch_new;

SELECT count(*) FROM orders;

1 rows affected.

count(*)
150000000


### OR 

use below if you want visually polished output 

In [26]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the workspace dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("USE tpch_branch_new")
        cur.execute("SELECT DATABASE()");             db = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM orders"); count = cur.fetchone()[0]

pt = PrettyTable(['Database', 'Total Rows'])
pt.add_row([db, f"{count:,}"])
print(pt)

+-----------------+-------------+
|     Database    |  Total Rows |
+-----------------+-------------+
| tpch_branch_new | 150,000,000 |
+-----------------+-------------+


### Performance tuning with Database Branching
You can create a separate DB branch for performance testing and apply the modifications to the newly created branch before implementing them in production.

###### Example 1. Pricing Summary Report:
This query reports the total number of items, their prices, and average quantities shipped within 90 days of the most recent ship date (specified date) in the database.


In [ ]:
%%sql
USE tpch;
SELECT
    l_returnflag,
    l_linestatus,
    SUM(l_quantity) AS sum_qty,
    SUM(l_extendedprice) AS sum_base_price,
    SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
    SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
    AVG(l_quantity) AS avg_qty,
    AVG(l_extendedprice) AS avg_price,
    AVG(l_discount) AS avg_disc,
    COUNT(*) AS count_order
FROM
    lineitem
WHERE
    l_shipdate <= date('1998-12-01' - interval '90' day)
GROUP BY
    l_returnflag,
    l_linestatus
ORDER BY
    l_returnflag,
    l_linestatus;